In [3]:
import importlib
import DataMapArchitect as dma
from DataMapArchitect import DataMapArchitect
import custom_image_selecter as cis

importlib.reload(dma)
importlib.reload(cis)


<module 'custom_image_selecter' from 'c:\\Users\\ThinkPad\\Documents\\UNIVERSITY\\PFE2\\custom_image_selecter.py'>

In [ ]:
 # CONFIGURATION
DATA_PATH = "./data/CASIA_FASD_V3/DATA"
OUTPUT_JSON = [
    "./CASIA_FASD_V3_30percent_BMSelection.json",
    "./CASIA_FASD_V3_50percent_BMSelection.json",
    "./CASIA_FASD_V3_70percent_BMSelection.json",
    "./CASIA_FASD_V3_90percent_BMSelection.json",
]

# datamape creation

In [ ]:
import json
file = "./CASIA_FASD_V3_224-224_10percent_BCMSelection"
with open(file + ".json", "r") as f:
    data_map = json.load(f)
data_map

In [ ]:
# Initialize Architect
architect = DataMapArchitect()
r = 0.3
for output in OUTPUT_JSON:
    results = architect.create_map_parallel(
        dataroute=DATA_PATH,
        max_workers=8,
        keep_ratio=r, 
        filter_fn=dma.brightness_contrast_movement_based_selecting,
        image_selection_config=None
    )
    architect.save_to_json(output, results)
    r += 0.2

Mapping Progress: 100%|██████████| 600/600 [55:00<00:00,  5.50s/it]  



[SUCCESS] Map saved to: ./CASIA_FASD_V3_30percent_BMSelection.json


Mapping Progress: 100%|██████████| 600/600 [54:04<00:00,  5.41s/it]  



[SUCCESS] Map saved to: ./CASIA_FASD_V3_50percent_BMSelection.json


Mapping Progress:   6%|▌         | 33/600 [03:41<1:03:24,  6.71s/it]


# data pipline

In [ ]:
import DataLoaderPipeline as dplmodule
from DataLoaderPipeline import DataLoaderPipeline
import importlib
importlib.reload(dplmodule)

In [ ]:
DATA_PATH = "data/CASIA_FASD_V3/DATA"
DATA_MAP_PATH = "./CASIA_FASD_V3_224-224_10percent_BCMSelection.json"
all_subjects = [f"{i:02d}" for i in range(1, 51)]  # Generate subject IDs from '01' to '50'.
train_subs = all_subjects[:20]
val_subs = all_subjects[20:25] 
test_subs = all_subjects[20:] 

dlp = DataLoaderPipeline(data_map_path=DATA_MAP_PATH, data_path=DATA_PATH, pix_range=(-1.0,1.0), img_size=(224,224), batch_size=32)


train_ds = dlp.build_pipeline(train_subs, balanced=True, augment=True)
val_ds   = dlp.build_pipeline(val_subs, balanced=False, augment=False)
test_ds  = dlp.build_pipeline(test_subs, balanced=False, augment=False)
for ds in [train_ds, val_ds, test_ds]:
    dlp.audit_dataset(ds, batchs=20)

In [ ]:
dlp.data_path

In [ ]:
train_dict ={
    "dataMapPaths" : [
        "./CASIA_FASD_V3_224-224_10percent_BCMSelection.json",
        "./CASIA_FASD_V3_224-224_10percent_uniformSampling.json"
    ],
    "imageSizes" : [
        (224,224),
        (128,128)
    ],
    "pixelRanges" : [
        (-1.0,1.0),
        (-1.0,1.0),
        (0.0,255.0)
    ],
    "Augmentations" : [
        
    ]
}

In [ ]:
pixel_ranges = [(0.0,255.0), (0.0,1.0),(-1.0,1.0), (-1.0,1.0),(0.0,1.0),(0.0,1.0)]
filter = ["no_filter","no_filter" ,"BCMSelection","uniform_sampling" ,"BCMSelection","uniform_sampling"]
eer = [0.0455, 0.0334, 0.0398, 0.234, 0.0672,0.138]
accuracy = [0.96,0.96, 0.94, 0.83, 0.93, 0.78]

# show table
import pandas as pd
# align left and set column width
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.width', 1200)


df = pd.DataFrame({
    "Pixel Range": pixel_ranges,
    "EER": eer,
    "Accuracy": accuracy,
    "Filter - 0.1 ratio": filter,
})
df


In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf
def create_custom_model(input_shape=(224, 224, 3)):
  model = models.Sequential(name="FASD_Custom_Net")
  
  model.add(layers.Input(shape=input_shape))

  model.add(layers.Conv2D(32, (3, 3), activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2, 2)))

  model.add(layers.Conv2D(64, (3, 3), activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2, 2)))

  model.add(layers.Conv2D(128, (3, 3), activation='relu'))
  model.add(layers.BatchNormalization())
  model.add(layers.MaxPooling2D((2, 2)))

  model.add(layers.Conv2D(224, (3, 3), activation='relu'))
  model.add(layers.MaxPooling2D((2, 2)))

  model.add(layers.Flatten())
  model.add(layers.Dense(256, activation='relu'))
  model.add(layers.Dropout(0.5))
  model.add(layers.Dense(1, activation='sigmoid')) 

  model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
      loss = tf.keras.losses.BinaryCrossentropy(),
      metrics=['accuracy']
  )

  return model